# VisA PCB4 이상 탐지 — 안정 실행형 제출 노트북
이 파일을 Windows의 C:\Users\USER\Desktop\ERICA_X_MODULABS_CV\07_Anomaly_detection 폴더에 넣고 JupyterLab 메뉴 **Run → Run All Cells**를 실행하세요. 기존 VisA 폴더를 그대로 사용하며 데이터는 다운로드하지 않습니다.

원본 실행 오류 수정: Windows용 num_workers=0, 유효한 실제 정답 마스크 자동 탐색, 모델 재정의와 중복 학습 제거, ResNet18 CNN 특징 디스크 캐시, PaDiM 청크 통계(전체 특징 RAM 누적 없음), 정상 이미지 기준 임계값, 특징 차원 비교, 세 모델 성능 비교, 자동 결과 저장 및 검증.

**빠른 실행 기본 설정:** 정상 학습 이미지 최대 200장, 테스트는 정상 201장 + 불량 100장 전부. 전체 804장으로 학습하려면 MAX_TRAIN_IMAGES=None으로 변경하세요. AutoEncoder는 시간 절약을 위해 2 epoch이므로 충분히 학습된 모델과 동일시하면 안 됩니다. 인터넷 장애로 CNN 사전학습 가중치를 받지 못한 경우 계속 실행하지만 신뢰 제한을 결과에 기록합니다. 장시간 셀은 진행률을 보여 줍니다.

In [ ]:
# 01 | 필수 패키지 자동 점검
import os,sys,time,json,random,hashlib,importlib.util,subprocess
from pathlib import Path
for mod,pkg in [("numpy","numpy"),("pandas","pandas"),("PIL","pillow"),("matplotlib","matplotlib"),("sklearn","scikit-learn"),("tqdm","tqdm")]:
    if importlib.util.find_spec(mod) is None:
        print("설치:",pkg,flush=True)
        subprocess.check_call([sys.executable,"-m","pip","install",pkg])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torchvision import models,transforms
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import roc_auc_score,average_precision_score,roc_curve,confusion_matrix,f1_score,precision_score,recall_score
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
print("01 OK | Python",sys.version.split()[0],"| torch",torch.__version__,"| cuda",torch.cuda.is_available(),flush=True)

In [ ]:
# 02 | 프로젝트 자동 위치 확인 + 사용자 설정
DEFAULT=Path(r"C:\Users\USER\Desktop\ERICA_X_MODULABS_CV\07_Anomaly_detection")
candidates=[DEFAULT,Path.cwd(),Path.cwd()/"07_Anomaly_detection",Path.home()/"Desktop"/"ERICA_X_MODULABS_CV"/"07_Anomaly_detection"]
ROOT=next((p.resolve() for p in candidates if (p/"VisA"/"pcb4"/"image_anno.csv").is_file()),None)
if ROOT is None: raise FileNotFoundError("VisA/pcb4/image_anno.csv가 없습니다. 이 파일을 07_Anomaly_detection 폴더에 넣어 주세요.")
os.chdir(ROOT)
CSV=ROOT/"VisA"/"pcb4"/"image_anno.csv"
OUT=ROOT/"padim_outputs"
CACHE=OUT/"feature_cache"
CACHE.mkdir(parents=True,exist_ok=True)
SEED=42
IMAGE_SIZE=192
GRID=14
MAX_TRAIN_IMAGES=200  # 804장 전체를 학습하려면 None
AE_EPOCHS=2
DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH=8 if DEVICE.type=="cuda" else 4
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if DEVICE.type=="cpu": torch.set_num_threads(min(4,os.cpu_count() or 1))
print("02 OK | 폴더:",ROOT,"| 장치:",DEVICE,"| 실제 학습 이미지 상한:",MAX_TRAIN_IMAGES,flush=True)

In [ ]:
# 03 | VisA CSV 직접 로드: 원본 분할(정상 80%/20% + 불량 전체)
df=pd.read_csv(CSV)
required={"image","label","mask"}
if not required.issubset(df.columns):raise ValueError("CSV 필수 컬럼이 없습니다: "+str(required-df.columns.to_series().index.to_set()))
df["label"]=df["label"].astype(str).str.lower().str.strip()
normal=df[df.label=="normal"]
bad=df[df.label!="normal"]
cut=int(len(normal)*.8)
def resolve(s):
    if pd.isna(s):return None
    s=str(s).strip().replace("\\","/").lstrip("./")
    if not s:return None
    p=Path(s)
    options=[p] if p.is_absolute() else [ROOT/"VisA"/p,ROOT/"VisA"/"pcb4"/p]
    return next((q.resolve() for q in options if q.is_file()),None)
def records(frame):
    output=[]
    for _,r in frame.iterrows():
        img=resolve(r["image"])
        if img is None:raise FileNotFoundError("이미지 경로 오류: "+str(r["image"]))
        output.append({"image":img,"mask":resolve(r["mask"]),"label":int(r["label"]!="normal"),"type":r["label"]})
    return output
train_full=records(normal.iloc[:cut])
test_rows=records(pd.concat([normal.iloc[cut:],bad],ignore_index=True))
if MAX_TRAIN_IMAGES is not None and len(train_full)>MAX_TRAIN_IMAGES:
    indexes=sorted(np.random.default_rng(SEED).choice(len(train_full),MAX_TRAIN_IMAGES,replace=False))
    train_rows=[train_full[i] for i in indexes]
else:train_rows=train_full
labels=np.array([r["label"] for r in test_rows],dtype=int)
assert len(train_rows)>=3 and (labels==0).any() and (labels==1).any()
print("03 OK | 정상 전체",len(normal),"| 불량 전체",len(bad))
print("정상 학습 분할",len(train_full),"| 실제 학습",len(train_rows),"| 테스트 정상",int((labels==0).sum()),"| 테스트 불량",int((labels==1).sum()),flush=True)

In [ ]:
# 04 | DataLoader(num_workers=0) 및 GT 마스크 검사
transform=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class VisaDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        with Image.open(self.rows[i]["image"]) as im:
            x=transform(im.convert("RGB"))
        return {"image":x,"label":self.rows[i]["label"]}
train_data=VisaDataset(train_rows);test_data=VisaDataset(test_rows)
train_loader=DataLoader(train_data,batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=False)
test_loader=DataLoader(test_data,batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=False)
sample_batch=next(iter(test_loader))
assert tuple(sample_batch["image"].shape[1:])==(3,IMAGE_SIZE,IMAGE_SIZE)
valid_mask_row=None
for r in test_rows:
    if r["label"]==0 or r["mask"] is None:continue
    try:
        with Image.open(r["mask"]) as im: pix=np.asarray(im.convert("L"))
        if np.any(pix>0):
            valid_mask_row=r;break
    except (OSError,ValueError):pass
if valid_mask_row:
    with Image.open(valid_mask_row["image"]) as im: raw=np.asarray(im.convert("RGB").resize((IMAGE_SIZE,IMAGE_SIZE)))
    with Image.open(valid_mask_row["mask"]) as im: mask=np.asarray(im.convert("L").resize((IMAGE_SIZE,IMAGE_SIZE),Image.NEAREST))>0
    overlay=raw.astype(np.float32)/255
    overlay[mask]=.4*overlay[mask]+.6*np.array([1.,0.,0.],dtype=np.float32)
    fig,axs=plt.subplots(1,3,figsize=(12,4))
    for ax,img,title in zip(axs,[raw,mask,overlay],["Original","Ground Truth Mask","Overlay"]):
        ax.imshow(img,cmap="gray" if title=="Ground Truth Mask" else None);ax.set_title(title);ax.axis("off")
    fig.tight_layout();fig.savefig(OUT/"ground_truth_preview.png",dpi=130);plt.show();plt.close(fig)
    print("마스크 정상 | 실제 결함 픽셀:",int(mask.sum()))
else:print("경고: 비어 있지 않은 정답 마스크를 찾지 못했습니다. 빈 마스크를 결함으로 꾸미지 않습니다.")
print("04 OK | batch",tuple(sample_batch["image"].shape),"| workers=0",flush=True)

In [ ]:
# 05 | 사전학습 ResNet18 고정 특징 추출기. 실패 시 신뢰 제한을 분명히 표시.
WEIGHT_STATUS="pretrained"
try:
    backbone=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
except Exception as e:
    WEIGHT_STATUS="untrained_fallback"
    print("주의: ImageNet 사전학습 가중치 다운로드/로드 실패:",str(e)[:200])
    print("무작위 CNN으로 진행하며 이 실행의 성능은 제출용 확정 지표가 아닙니다.")
    backbone=models.resnet18(weights=None)
backbone=backbone.to(DEVICE).eval()
for p in backbone.parameters():p.requires_grad_(False)
class Extractor(nn.Module):
    def __init__(self,net):super().__init__();self.net=net
    def forward(self,x):
        m=self.net
        x=m.maxpool(m.relu(m.bn1(m.conv1(x))))
        a=m.layer1(x);b=m.layer2(a);c=m.layer3(b)
        z=torch.cat([F.adaptive_avg_pool2d(v,(GRID,GRID)) for v in [a,b,c]],dim=1)
        return z.permute(0,2,3,1).reshape(z.shape[0],GRID*GRID,448)
extractor=Extractor(backbone).to(DEVICE).eval()
with torch.inference_mode():check=extractor(sample_batch["image"][:1].to(DEVICE))
assert tuple(check.shape)==(1,GRID*GRID,448)
del check
print("05 OK | ResNet18 특징 448채널 | patches",GRID*GRID,"|",WEIGHT_STATUS,flush=True)

In [ ]:
# 06 | 특징을 매 배치마다 디스크 .npy에 저장: 메모리 폭증 방지
from numpy.lib.format import open_memmap
def cache_features(rows,loader,name):
    signature="|".join(str(r["image"]) for r in rows)+f"|{IMAGE_SIZE}|{GRID}|{WEIGHT_STATUS}"
    digest=hashlib.sha256(signature.encode("utf-8")).hexdigest()[:12]
    filename=CACHE/f"{name}_{digest}.npy"
    flag=CACHE/f"{name}_{digest}.done"
    shape=(len(rows),GRID*GRID,448)
    if flag.is_file() and filename.is_file():
        try:
            prev=np.load(filename,mmap_mode="r")
            if prev.shape==shape and prev.dtype==np.float32:
                print("기존 캐시 재사용:",filename.name,flush=True);return filename
        except (OSError,ValueError):pass
    if flag.exists():flag.unlink()
    mm=open_memmap(filename,mode="w+",dtype="float32",shape=shape)
    offset=0
    for batch in tqdm(loader,desc=f"{name} CNN 특징 추출"):
        with torch.inference_mode():
            arr=extractor(batch["image"].to(DEVICE)).cpu().numpy()
        n=len(arr);mm[offset:offset+n]=arr;offset+=n
    assert offset==len(rows)
    mm.flush();del mm
    flag.write_text("complete",encoding="utf-8")
    print("캐시 저장:",filename.name,flush=True)
    return filename
started=time.perf_counter()
train_file=cache_features(train_rows,train_loader,"train")
test_file=cache_features(test_rows,test_loader,"test")
train_features=np.load(train_file,mmap_mode="r")
test_features=np.load(test_file,mmap_mode="r")
assert train_features.shape[1:]==test_features.shape[1:]==(GRID*GRID,448)
print("06 OK | train",train_features.shape,"| test",test_features.shape,"| elapsed",round(time.perf_counter()-started,1),"초",flush=True)

In [ ]:
# 07 | 배치 통계 기반 PaDiM: 메모리맵으로부터 패치별 평균/공분산 산출
def fit_padim(source,dim):
    n,p,c=source.shape
    ids=np.sort(np.random.default_rng(SEED).choice(c,dim,replace=False))
    sm=np.zeros((p,dim),np.float64)
    xx=np.zeros((p,dim,dim),np.float64)
    for i in range(0,n,16):
        batch=np.asarray(source[i:i+16],dtype=np.float32)[:,:,ids].astype(np.float64)
        sm+=batch.sum(axis=0)
        xx+=np.einsum("bpi,bpj->pij",batch,batch,optimize=True)
    mean=sm/n
    cov=(xx-n*np.einsum("pi,pj->pij",mean,mean))/(n-1)
    cov+=.01*np.eye(dim)[None,:,:]
    inv=np.linalg.pinv(cov)
    if not np.isfinite(inv).all():raise RuntimeError("공분산 역행렬에 비정상값 존재")
    return {"ids":ids,"mean":mean,"inv":inv}
def predict_padim(source,model):
    n,p,c=source.shape
    scores=np.zeros(n,np.float64);maps=np.zeros((n,GRID,GRID),np.float32)
    for i in range(0,n,16):
        b=np.asarray(source[i:i+16],dtype=np.float32)[:,:,model["ids"]].astype(np.float64)
        diff=b-model["mean"][None]
        dist=np.sqrt(np.maximum(np.einsum("bpi,pij,bpj->bp",diff,model["inv"],diff,optimize=True),0))
        k=len(b);scores[i:i+k]=dist.max(axis=1);maps[i:i+k]=dist.reshape(k,GRID,GRID)
    if not np.isfinite(scores).all():raise RuntimeError("점수에 NaN/Inf 발생")
    return scores,maps
print("PaDiM 학습 시작 (청크 통계)")
padim=fit_padim(train_features,dim=48)
padim_train,_=predict_padim(train_features,padim)
padim_test,padim_maps=predict_padim(test_features,padim)
threshold=float(np.percentile(padim_train,99))
np.savez_compressed(OUT/"padim_model_state.npz",ids=padim["ids"],mean=padim["mean"],inv=padim["inv"],threshold=threshold,grid=GRID,image_size=IMAGE_SIZE,weights=WEIGHT_STATUS)
print("07 OK | PaDiM 완료 | train",len(padim_train),"| test",len(padim_test),"| threshold",round(threshold,4),flush=True)

In [ ]:
# 08 | AUROC/AUPRC/F1 및 혼동행렬(임계값은 학습 정상 점수 p99)
def evaluate(y,s,thr):
    yhat=(np.asarray(s)>=thr).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,yhat,labels=[0,1]).ravel()
    return dict(AUROC=float(roc_auc_score(y,s)),AUPRC=float(average_precision_score(y,s)),F1=float(f1_score(y,yhat,zero_division=0)),Precision=float(precision_score(y,yhat,zero_division=0)),Recall=float(recall_score(y,yhat,zero_division=0)),TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),Threshold=float(thr))
results={"PaDiM":evaluate(labels,padim_test,threshold)}
scores_by_method={"PaDiM":padim_test}
display(pd.DataFrame.from_dict(results,orient="index").round(4))
print("08 OK | PaDiM AUROC:",round(results["PaDiM"]["AUROC"],4),"| FP/FN",results["PaDiM"]["FP"],results["PaDiM"]["FN"],flush=True)

In [ ]:
# 09 | 추가 실험 1: 정상 학습 점수 기준 임계값과 FP/FN 변화
rows=[]
for q in (90,95,97,99,99.5):
    t=float(np.percentile(padim_train,q))
    rows.append({"Percentile":q,**evaluate(labels,padim_test,t)})
threshold_df=pd.DataFrame(rows)
threshold_df.to_csv(OUT/"threshold_experiment.csv",index=False,encoding="utf-8-sig")
display(threshold_df[["Percentile","TP","FP","TN","FN","Precision","Recall","F1"]].round(4))
fig,ax=plt.subplots(figsize=(8,4))
for metric in ("Recall","F1"):ax.plot(threshold_df["Percentile"],threshold_df[metric],marker="o",label=metric)
ax.set(title="Threshold experiment",xlabel="Train-normal percentile",ylabel="Metric",ylim=(0,1.03));ax.grid(alpha=.3);ax.legend()
fig.tight_layout();fig.savefig(OUT/"threshold_experiment.png",dpi=140);plt.show();plt.close(fig)
print("09 OK | 임계값 표/그림 저장",flush=True)

In [ ]:
# 10 | 추가 실험 2: 동일 캐시로 특징 차원 32 vs 64 (CNN 재추출 없음)
dimension_rows=[]
for dim in (32,64):
    started=time.perf_counter()
    model=fit_padim(train_features,dim=dim)
    train_scores,_=predict_padim(train_features,model)
    test_scores,_=predict_padim(test_features,model)
    metrics=evaluate(labels,test_scores,float(np.percentile(train_scores,99)))
    dimension_rows.append({"Dimension":dim,"Time_seconds":round(time.perf_counter()-started,2),**metrics})
    print("dim",dim,"| AUROC:",round(metrics["AUROC"],4),"| 시간:",dimension_rows[-1]["Time_seconds"],"초",flush=True)
dimension_df=pd.DataFrame(dimension_rows)
dimension_df.to_csv(OUT/"dimension_experiment.csv",index=False,encoding="utf-8-sig")
display(dimension_df[["Dimension","AUROC","AUPRC","F1","Time_seconds"]].round(4))
fig,ax=plt.subplots(figsize=(7,4))
for metric in ("AUROC","AUPRC"):ax.plot(dimension_df.Dimension,dimension_df[metric],marker="o",label=metric)
ax.set(xlabel="Reduced feature dimension",ylabel="Metric",title="PaDiM dimension comparison",ylim=(0,1.03))
ax.grid(alpha=.3);ax.legend();fig.tight_layout();fig.savefig(OUT/"dimension_experiment.png",dpi=140);plt.show();plt.close(fig)
print("10 OK | 특징 차원 표/그림 저장",flush=True)

In [ ]:
# 11 | 기준 모델 One-Class SVM (정상 이미지 전역 CNN 특징으로 학습)
def mean_features(mm):
    out=np.zeros((len(mm),mm.shape[-1]),dtype=np.float32)
    for i in range(0,len(mm),16):
        batch=np.asarray(mm[i:i+16],dtype=np.float32)
        out[i:i+len(batch)]=batch.mean(axis=1)
    return out
scaler=StandardScaler()
svm=OneClassSVM(nu=.1,kernel="rbf",gamma="scale")
svm_train=mean_features(train_features)
svm_test=mean_features(test_features)
svm.fit(scaler.fit_transform(svm_train))
svm_normal_scores=-svm.decision_function(scaler.transform(svm_train)).ravel()
svm_scores=-svm.decision_function(scaler.transform(svm_test)).ravel()
results["One-Class SVM"]=evaluate(labels,svm_scores,float(np.percentile(svm_normal_scores,99)))
scores_by_method["One-Class SVM"]=svm_scores
print("11 OK | SVM AUROC:",round(results["One-Class SVM"]["AUROC"],4),flush=True)

In [ ]:
# 12 | 경량 AutoEncoder (기본 2 epoch, 실습용 비교)
class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Conv2d(3,16,4,2,1),nn.ReLU(),nn.Conv2d(16,32,4,2,1),nn.ReLU(),nn.Conv2d(32,64,4,2,1),nn.ReLU())
        self.decoder=nn.Sequential(nn.ConvTranspose2d(64,32,4,2,1),nn.ReLU(),nn.ConvTranspose2d(32,16,4,2,1),nn.ReLU(),nn.ConvTranspose2d(16,3,4,2,1),nn.Sigmoid())
    def forward(self,x):return self.decoder(self.encoder(x))
MEAN=torch.tensor([.485,.456,.406]).view(1,3,1,1).to(DEVICE)
STD=torch.tensor([.229,.224,.225]).view(1,3,1,1).to(DEVICE)
def input_ae(x):
    rgb=(x*STD+MEAN).clamp(0,1)
    return F.interpolate(rgb,size=(64,64),mode="bilinear",align_corners=False)
ae=SmallAE().to(DEVICE)
opt=torch.optim.Adam(ae.parameters(),lr=.001)
ae_loader=DataLoader(train_data,batch_size=BATCH,shuffle=True,num_workers=0)
for epoch in range(AE_EPOCHS):
    ae.train();losses=[]
    for batch in tqdm(ae_loader,desc=f"AE epoch {epoch+1}/{AE_EPOCHS}"):
        x=input_ae(batch["image"].to(DEVICE))
        pred=ae(x);loss=F.mse_loss(pred,x)
        opt.zero_grad();loss.backward();opt.step()
        losses.append(loss.item())
    print("AE",epoch+1,"/",AE_EPOCHS,"loss",round(float(np.mean(losses)),6),flush=True)
ae.eval()
def infer_ae(loader):
    ans=[]
    with torch.inference_mode():
        for batch in loader:
            x=input_ae(batch["image"].to(DEVICE));rec=ae(x)
            ans.extend((rec-x).pow(2).mean(dim=(1,2,3)).cpu().tolist())
    return np.array(ans)
ae_train=infer_ae(train_loader);ae_test=infer_ae(test_loader)
results["AutoEncoder (2 epochs)"]=evaluate(labels,ae_test,float(np.percentile(ae_train,99)))
scores_by_method["AutoEncoder (2 epochs)"]=ae_test
torch.save(ae.cpu().state_dict(),OUT/"autoencoder_weights.pt");ae.to(DEVICE)
print("12 OK | AE AUROC:",round(results["AutoEncoder (2 epochs)"]["AUROC"],4),flush=True)

In [ ]:
# 13 | 그래프/결과 자동 저장
comparison=pd.DataFrame.from_dict(results,orient="index")
comparison.index.name="Method"
comparison.to_csv(OUT/"model_comparison.csv",encoding="utf-8-sig")
display(comparison.round(4))
fig,ax=plt.subplots(figsize=(8,5))
for name,s in scores_by_method.items():
    fpr,tpr,_=roc_curve(labels,s)
    ax.plot(fpr,tpr,label=f"{name} (AUC={results[name]['AUROC']:.3f})")
ax.plot([0,1],[0,1],"k--",label="Random");ax.set(xlabel="FPR",ylabel="TPR",title="Image-level ROC")
ax.grid(alpha=.3);ax.legend(fontsize=8);fig.tight_layout();fig.savefig(OUT/"roc_comparison.png",dpi=140);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(padim_test[labels==0],bins=25,alpha=.6,label="Normal")
ax.hist(padim_test[labels==1],bins=25,alpha=.6,label="Anomaly")
ax.axvline(threshold,color="black",ls="--",label="Train p99 threshold")
ax.set(title="PaDiM anomaly score distribution",xlabel="Max Mahalanobis distance",ylabel="Images");ax.legend()
fig.tight_layout();fig.savefig(OUT/"score_distribution.png",dpi=140);plt.show();plt.close(fig)
i=int(np.where(labels==1)[0][0])
with Image.open(test_rows[i]["image"]) as im:base=np.asarray(im.convert("RGB").resize((IMAGE_SIZE,IMAGE_SIZE)))
small=padim_maps[i]
big=np.asarray(Image.fromarray(small.astype(np.float32)).resize((IMAGE_SIZE,IMAGE_SIZE),Image.BILINEAR))
scale=(big-big.min())/(np.ptp(big)+1e-8)
heat=(plt.get_cmap("jet")(scale)[...,:3]*255).astype(np.uint8)
merged=np.clip(.6*base.astype(np.float32)+.4*heat.astype(np.float32),0,255).astype(np.uint8)
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,img,title in zip(axs,[base,heat,merged],["Original","Predicted anomaly heatmap","Overlay"]):
    ax.imshow(img);ax.set_title(title);ax.axis("off")
fig.tight_layout();fig.savefig(OUT/"predicted_anomaly_map.png",dpi=140);plt.show();plt.close(fig)
print("13 OK | 모델 비교표/ROC/점수분포/이상맵 저장",flush=True)

In [ ]:
# 14 | 과제 제출용 결과 보고서 자동 작성 + 산출물 검수
lines=["# VisA PCB4 이상 탐지 실습 결과","",f"- 학습 정상: {len(train_rows)}장 / 전체 학습 분할 {len(train_full)}장",
       f"- 테스트 정상: {(labels==0).sum()}장 / 불량: {(labels==1).sum()}장",
       f"- ResNet18 상태: {WEIGHT_STATUS} / 이미지 {IMAGE_SIZE} / GRID {GRID}",
       f"- AE epoch: {AE_EPOCHS}","","## 이미지 단위 평가",
       "| Method | AUROC | AUPRC | F1 | FP | FN |","|---|---:|---:|---:|---:|---:|"]
for name,r in results.items():
    lines.append(f"| {name} | {r['AUROC']:.4f} | {r['AUPRC']:.4f} | {r['F1']:.4f} | {r['FP']} | {r['FN']} |")
lines += ["","## 추가 실험","- Threshold: 정상 학습 점수 백분위 90,95,97,99,99.5에 따른 FP/FN 비교. threshold_experiment.csv/png 참조.",
          "- Feature dimension: 32 vs 64 차원 비교. dimension_experiment.csv/png 참조.",
          "","## 한계/해석",
          "- 기본 실행은 학습 분할 전체가 아니라 일부 정상 영상으로 학습하여 전체 804장 조건과 직접 비교할 수 없습니다.",
          "- 임계값은 학습 정상 점수로 계산했고 별도 검증 세트 사용이 권장됩니다.",
          "- AutoEncoder 2 epoch은 빠른 비교용이며 최적 성능을 보장하지 않습니다.",
          "- AUROC/AUPRC는 이미지 수준이며 픽셀 수준 마스크 평가가 아닙니다."]
if WEIGHT_STATUS!="pretrained":lines.append("- 주의: 무작위 ResNet18 가중치로 실행했으므로 성능을 신뢰할 수 있는 최종 값으로 주장하면 안 됩니다.")
(OUT/"result_report.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
files=["padim_model_state.npz","threshold_experiment.csv","threshold_experiment.png","dimension_experiment.csv","dimension_experiment.png",
       "autoencoder_weights.pt","model_comparison.csv","roc_comparison.png","score_distribution.png","predicted_anomaly_map.png","result_report.md"]
if valid_mask_row:files+=["ground_truth_preview.png"]
missing=[f for f in files if not (OUT/f).is_file()]
assert not missing,"산출물 누락: "+str(missing)
assert len(padim_test)==len(labels) and np.isfinite(padim_maps).all()
for key,r in results.items():
    assert 0<=r["AUROC"]<=1 and 0<=r["AUPRC"]<=1 and 0<=r["F1"]<=1,key
print("="*50)
print("14 전체 실험 실행 및 검수 완료 | 결과 폴더:",OUT)
print("모델:",len(results),"| 테스트:",len(labels),"| 산출물:",len(files))
if WEIGHT_STATUS!="pretrained":print("경고: 사전학습 가중치 사용 실패 — 점수 해석 주의")
print("Jupyter Ctrl+S로 실행된 노트북을 저장하세요.")
print("="*50,flush=True)

## 선택: Gradio 실시간 추론
제출에 필요한 모든 결과는 위 셀에서 이미 저장됩니다. 아래 셀은 기본 실행 시 메시지만 출력하며 서버를 켜지 않습니다. 나중에 UI가 필요하면 ENABLE_GRADIO=True로 바꿔 다시 실행하세요. 외부 공유는 사용하지 않습니다.

In [ ]:
# 15 | 옵션: 공개 링크 없는 Gradio
ENABLE_GRADIO=False
if ENABLE_GRADIO:
    try:
        import gradio as gr
        def infer(pil):
            with torch.inference_mode():
                x=transform(pil.convert("RGB")).unsqueeze(0).to(DEVICE)
                feats=extractor(x).cpu().numpy()[:,:,padim["ids"]].astype(np.float64)
            diff=feats-padim["mean"][None]
            dist=np.sqrt(np.maximum(np.einsum("bpi,pij,bpj->bp",diff,padim["inv"],diff,optimize=True),0))
            value=float(dist.max())
            mini=dist[0].reshape(GRID,GRID).astype(np.float32)
            big=np.asarray(Image.fromarray(mini).resize((IMAGE_SIZE,IMAGE_SIZE),Image.BILINEAR))
            scaled=(big-big.min())/(np.ptp(big)+1e-8)
            cmap=(plt.get_cmap("jet")(scaled)[...,:3]*255).astype(np.uint8)
            src=np.asarray(pil.convert("RGB").resize((IMAGE_SIZE,IMAGE_SIZE)))
            overlay=np.clip(.6*src+.4*cmap,0,255).astype(np.uint8)
            label="Anomaly" if value>=threshold else "Normal"
            return f"{label} | score={value:.4f} | threshold={threshold:.4f}",cmap,overlay
        gr.Interface(infer,inputs=gr.Image(type="pil"),outputs=[gr.Textbox(),gr.Image(type="numpy"),gr.Image(type="numpy")],title="PCB4 PaDiM anomaly detection").launch(share=False,prevent_thread_lock=True)
    except Exception as exc:print("Gradio 선택 기능 실패(핵심 결과 이미 저장):",repr(exc))
else:print("15 OK | Gradio 자동 실행 생략 — 제출 결과는 이미 생성됐습니다.")